# 15_train_B1_P2_full.ipynb
## Tuần 7–8: Huấn Luyện Đầy Đủ Mô Hình B1 (YOLOv8n + P2)

### 1. Mục tiêu
- **Huấn luyện hoàn chỉnh B1 ($YOLOv8n + P_2$)**: Huấn luyện mô hình kiến trúc có tầng phát hiện vi mô $P_2$ trên toàn bộ tập dữ liệu huấn luyện `train.txt` (3,989 ảnh).
- **Đảm bảo so sánh công bằng tuyệt đối với Baseline B0**:
  - Giữ nguyên số epoch: cùng 20 epochs như $B_0$.
  - Giữ nguyên độ phân giải ảnh: `imgsz: 640`.
  - Giữ nguyên batch size: `batch: 16` (trừ khi tràn bộ nhớ VRAM, sẽ ghi nhận rõ).
  - Giữ nguyên bộ tạo số ngẫu nhiên: `seed: 41`.
  - Giữ nguyên công thức tăng cường dữ liệu (Augmentation recipe đã khóa cho PCB: `mosaic: 1.0`, `fliplr: 0.5`, `degrees: 0.0`, v.v.).
  - Giữ nguyên hàm tối ưu hóa (`optimizer: auto`) và hàm mất mát (`CIoU + DFL + BCE`).
- **Nguyên tắc cô lập biến (Isolation of Variables)**:
  - Biến can thiệp duy nhất: Cấu trúc mô hình được tích hợp tầng $P_2$ (stride 4, $160 \times 160$).
  - **Tuyệt đối không dùng tập test (`test.txt`) để chọn epoch, chọn checkpoint hay tuning siêu tham số**.

### 2. Import thư viện
Nạp các thư viện chuẩn theo quy chuẩn kỹ thuật: `pathlib`, `yaml`, `json`, `shutil`, `subprocess`, `datetime`, `time`, `pandas`, `ultralytics.YOLO`.

In [7]:
from datetime import datetime, timezone
import json
import os
from pathlib import Path
import shutil
import subprocess
import time
import pandas as pd
import torch
from ultralytics import YOLO
import yaml

def find_project_root() -> Path:
    candidate = Path.cwd().resolve()
    for parent in [candidate, *candidate.parents]:
        if (parent / "configs").is_dir() and (parent / "data").is_dir():
            return parent
    return candidate

ROOT = find_project_root()
os.chdir(ROOT)
print(f"Project root     : {ROOT}")
print(f"Working directory: {os.getcwd()}")
print(f"PyTorch version  : {torch.__version__}, CUDA: {torch.cuda.is_available()}, MPS: {torch.backends.mps.is_available()}")

Project root     : /Users/Project/kltn-pcb
Working directory: /Users/Project/kltn-pcb
PyTorch version  : 2.14.0, CUDA: False, MPS: True


### 3. Load cấu hình thực nghiệm
- Nạp cấu hình tham chiếu: `configs/base.yaml`
- Nạp cấu hình dữ liệu: `data/data.yaml`
- Nạp định nghĩa kiến trúc B1: `configs/yolov8n-p2.yaml`

In [8]:
BASE_CONFIG_PATH = ROOT / "configs" / "base.yaml"
DATA_YAML_PATH = ROOT / "data" / "data.yaml"
P2_CONFIG_PATH = ROOT / "configs" / "yolov8n-p2.yaml"

with open(BASE_CONFIG_PATH, "r", encoding="utf-8") as f:
    base_cfg = yaml.safe_load(f)

with open(DATA_YAML_PATH, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

with open(P2_CONFIG_PATH, "r", encoding="utf-8") as f:
    p2_cfg = yaml.safe_load(f)

print("=== CẤU HÌNH THỰC NGHIỆM B1 (YOLOv8n-P2) ===")
print(f"Architecture YAML : {P2_CONFIG_PATH.relative_to(ROOT)}")
print(f"Dataset YAML      : {DATA_YAML_PATH.relative_to(ROOT)}")
print(f"Input imgsz       : {base_cfg.get('imgsz')}")
print(f"Epochs            : {base_cfg.get('epochs')}")
print(f"Batch size        : {base_cfg.get('batch')}")
print(f"Optimizer         : {base_cfg.get('optimizer')}")
print(f"Patience          : {base_cfg.get('patience')}")
print(f"Seed              : {base_cfg.get('seed')}")
print(f"Classes ({len(data_cfg.get('names', {}))}): {data_cfg.get('names')}")

=== CẤU HÌNH THỰC NGHIỆM B1 (YOLOv8n-P2) ===
Architecture YAML : configs/yolov8n-p2.yaml
Dataset YAML      : data/data.yaml
Input imgsz       : 640
Epochs            : 20
Batch size        : 16
Optimizer         : auto
Patience          : 30
Seed              : 41
Classes (6): {0: 'missing_hole', 1: 'mouse_bite', 2: 'open_circuit', 3: 'short', 4: 'spur', 5: 'spurious_copper'}


### 4. Pre-flight check (Kiểm tra điều kiện tiên quyết)
Kiểm tra nghiêm ngặt 8 điều kiện bắt buộc trước khi kích hoạt huấn luyện:
1. `results/leakage_report.txt` tồn tại và đạt `[PASS]`
2. `runs/B0/weights/best.pt` tồn tại (mốc baseline $B_0$ đã có)
3. `results/B0/metrics.json` tồn tại (kết quả thẩm định $B_0$ đã có)
4. `results/B1/architecture_cost.csv` tồn tại (đã phân tích chi phí phần cứng)
5. `runs/smoke_B1_P2/weights/best.pt` tồn tại (pipeline B1 đã pass smoke test)
6. `data/splits/train.txt` tồn tại
7. `data/splits/val.txt` tồn tại
8. `data/splits/split_manifest.json` tồn tại

In [9]:
LEAKAGE_REPORT = ROOT / "results" / "leakage_report.txt"
B0_BEST_PT = ROOT / "runs" / "B0" / "weights" / "best.pt"
B0_METRICS_JSON = ROOT / "results" / "B0" / "metrics.json"
B1_COST_CSV = ROOT / "results" / "B1" / "architecture_cost.csv"
B1_SMOKE_BEST = ROOT / "runs" / "smoke_B1_P2" / "weights" / "best.pt"
TRAIN_TXT = ROOT / "data" / "splits" / "train.txt"
VAL_TXT = ROOT / "data" / "splits" / "val.txt"
SPLIT_MANIFEST = ROOT / "data" / "splits" / "split_manifest.json"

# 1. Leakage report
assert LEAKAGE_REPORT.is_file(), f"Thiếu file: {LEAKAGE_REPORT}"
leakage_content = LEAKAGE_REPORT.read_text(encoding="utf-8")
assert "[PASS]" in leakage_content, "Leakage report không đạt trạng thái PASS!"
print("  [PASS] 1. Báo cáo rò rỉ dữ liệu (leakage_report.txt) đạt PASS.")

# 2. B0 best checkpoint
assert B0_BEST_PT.is_file(), f"Thiếu B0 checkpoint: {B0_BEST_PT}"
print(f"  [PASS] 2. Checkpoint B0 tồn tại ({B0_BEST_PT.relative_to(ROOT)}).")

# 3. B0 metrics
assert B0_METRICS_JSON.is_file(), f"Thiếu B0 metrics: {B0_METRICS_JSON}"
print(f"  [PASS] 3. Kết quả đánh giá B0 tồn tại ({B0_METRICS_JSON.relative_to(ROOT)}).")

# 4. B1 architecture cost
assert B1_COST_CSV.is_file(), f"Thiếu bảng chi phí kiến trúc B1: {B1_COST_CSV}"
print(f"  [PASS] 4. Bảng chi phí B1 tồn tại ({B1_COST_CSV.relative_to(ROOT)}).")

# 5. B1 smoke checkpoint
assert B1_SMOKE_BEST.is_file(), f"Thiếu B1 smoke checkpoint: {B1_SMOKE_BEST}"
print(f"  [PASS] 5. Checkpoint smoke B1 tồn tại ({B1_SMOKE_BEST.relative_to(ROOT)}).")

# 6, 7, 8. Split files
assert TRAIN_TXT.is_file(), f"Thiếu file: {TRAIN_TXT}"
assert VAL_TXT.is_file(), f"Thiếu file: {VAL_TXT}"
assert SPLIT_MANIFEST.is_file(), f"Thiếu file: {SPLIT_MANIFEST}"
print("  [PASS] 6, 7, 8. Đầy đủ các file phân vùng train/val và split_manifest.json.")

  [PASS] 1. Báo cáo rò rỉ dữ liệu (leakage_report.txt) đạt PASS.
  [PASS] 2. Checkpoint B0 tồn tại (runs/B0/weights/best.pt).
  [PASS] 3. Kết quả đánh giá B0 tồn tại (results/B0/metrics.json).
  [PASS] 4. Bảng chi phí B1 tồn tại (results/B1/architecture_cost.csv).
  [PASS] 5. Checkpoint smoke B1 tồn tại (runs/smoke_B1_P2/weights/best.pt).
  [PASS] 6, 7, 8. Đầy đủ các file phân vùng train/val và split_manifest.json.


### 5. Xác định ngân sách huấn luyện (Epoch Budget)
- Đọc `epochs` từ `configs/base.yaml`.
- Đối chiếu với lịch sử huấn luyện của $B_0$ trong `runs/B0/run_metadata.json` (hoặc `runs/B0/results.csv`).
- **Nguyên tắc công bằng**: Do $B_0$ đã huấn luyện chính xác **20 epochs**, mô hình $B_1$ **bắt buộc phải dùng đúng 20 epochs** để so sánh mAP, Precision, Recall và F1 một cách khách quan nhất.

In [10]:
B0_METADATA_PATH = ROOT / "runs" / "B0" / "run_metadata.json"
b0_epochs = None
if B0_METADATA_PATH.exists():
    with open(B0_METADATA_PATH, "r", encoding="utf-8") as f:
        b0_meta = json.load(f)
    b0_epochs = b0_meta.get("epochs")

cfg_epochs = int(base_cfg.get("epochs", 20))
if b0_epochs is not None:
    assert cfg_epochs == b0_epochs, f"LỖI: epochs cấu hình ({cfg_epochs}) lệch so với B0 ({b0_epochs})!"
    EPOCHS = b0_epochs
else:
    EPOCHS = cfg_epochs

print(f"XÁC NHẬN EPOCH BUDGET:")
print(f"  - Baseline B0 epochs : {b0_epochs}")
print(f"  - Config base epochs  : {cfg_epochs}")
print(f"  -> B1 Full Train sử dụng chính xác: {EPOCHS} epochs (Đảm bảo đối sánh công bằng).")

XÁC NHẬN EPOCH BUDGET:
  - Baseline B0 epochs : 20
  - Config base epochs  : 20
  -> B1 Full Train sử dụng chính xác: 20 epochs (Đảm bảo đối sánh công bằng).


### 6. Thiết lập môi trường thực thi (Run Setup)
- `run_name = "B1"`
- `run_dir = runs/B1`
- `result_dir = results/B1`
- Lưu cấu hình đầy đủ `runs/B1/config_used.yaml`
- Lưu git commit hash hiện tại `runs/B1/git_commit.txt`
- Khởi tạo `runs/B1/run_metadata.json` ghi nhận thời điểm bắt đầu

In [11]:
RUN_NAME = "B1"
RUNS_DIR = ROOT / "runs"
RUN_DIR = RUNS_DIR / RUN_NAME
RESULT_DIR = ROOT / "results" / RUN_NAME

RUN_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Lưu config_used.yaml
config_used_path = RUN_DIR / "config_used.yaml"
training_config = dict(base_cfg)
training_config["model"] = "configs/yolov8n-p2.yaml"
training_config["epochs"] = EPOCHS
with open(config_used_path, "w", encoding="utf-8") as f:
    yaml.dump(training_config, f, default_flow_style=False, sort_keys=False)
print(f"Đã lưu: {config_used_path}")

# 2. Lưu git commit
git_commit_path = RUN_DIR / "git_commit.txt"
try:
    commit_hash = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(ROOT)).decode("utf-8").strip()
except Exception:
    commit_hash = "unknown"
git_commit_path.write_text(commit_hash, encoding="utf-8")
print(f"Đã lưu: {git_commit_path} (commit: {commit_hash})")

# 3. Tạo run_metadata.json sơ bộ
meta_path = RUN_DIR / "run_metadata.json"
run_meta = {
    "experiment": RUN_NAME,
    "mode": "full_B1_P2",
    "start_time": datetime.now(timezone.utc).isoformat(),
    "end_time": None,
    "duration_seconds": None,
    "git_commit": commit_hash,
    "base_model": "configs/yolov8n-p2.yaml",
    "seed": int(base_cfg.get("seed", 41)),
    "epochs": EPOCHS,
    "imgsz": int(base_cfg.get("imgsz", 640)),
    "batch": int(base_cfg.get("batch", 16)),
    "augmentation_recipe": {
        "degrees": 0.0,
        "shear": 0.0,
        "perspective": 0.0,
        "flipud": 0.0,
        "fliplr": 0.5,
        "mosaic": 1.0,
        "hsv_h": 0.015,
        "hsv_s": 0.7,
        "hsv_v": 0.4,
        "scale": 0.5,
        "translate": 0.1
    }
}
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(run_meta, f, indent=2)
print(f"Đã khởi tạo: {meta_path}")

Đã lưu: /Users/Project/kltn-pcb/runs/B1/config_used.yaml
Đã lưu: /Users/Project/kltn-pcb/runs/B1/git_commit.txt (commit: 31667dedef749a9acccca5832c47868cb39cd9e6)
Đã khởi tạo: /Users/Project/kltn-pcb/runs/B1/run_metadata.json


### 7. Huấn luyện đầy đủ mô hình B1 (Full Training B1)
Kích hoạt quá trình huấn luyện mô hình $B_1$ trên toàn bộ tập dữ liệu:
- Kiến trúc: `configs/yolov8n-p2.yaml`
- Dữ liệu: `data/data.yaml`
- Siêu tham số cố định: `imgsz=640`, `batch=16`, `epochs=20`, `seed=41`, `optimizer=auto`, `patience=30`
- Augmentation recipe: Cố định tuyệt đối theo quyết định đã chốt trong `decision_log.md`

In [12]:
PRETRAINED_WEIGHTS = ROOT / "yolov8n.pt"
print(f"Khởi tạo B1 từ {P2_CONFIG_PATH.name} và nạp pretrained backbone từ {PRETRAINED_WEIGHTS.name}...")
model_b1 = YOLO(str(P2_CONFIG_PATH)).load(str(PRETRAINED_WEIGHTS))

BATCH_SIZE = int(base_cfg.get("batch", 16))
start_train_time = time.time()

try:
    print(f"=== BẮT ĐẦU HUẤN LUYỆN B1 FULL ({EPOCHS} EPOCHS, BATCH {BATCH_SIZE}) ===")
    train_results = model_b1.train(
        data=str(DATA_YAML_PATH),
        epochs=EPOCHS,
        imgsz=int(base_cfg.get("imgsz", 640)),
        batch=BATCH_SIZE,
        seed=int(base_cfg.get("seed", 41)),
        optimizer=str(base_cfg.get("optimizer", "auto")),
        patience=int(base_cfg.get("patience", 30)),
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        # Augmentation recipe đã khóa
        degrees=0.0,
        shear=0.0,
        perspective=0.0,
        flipud=0.0,
        fliplr=0.5,
        mosaic=1.0,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        scale=0.5,
        translate=0.1,
        plots=True
    )
    training_success = True
except RuntimeError as e:
    err_msg = str(e).lower()
    if "out of memory" in err_msg or "cuda" in err_msg or "mps" in err_msg:
        print(f"CẢNH BÁO TRÀN BỘ NHỚ VRAM: {e}")
        training_success = False
    else:
        raise e

duration_sec = time.time() - start_train_time
print(f"Thời gian huấn luyện: {duration_sec:.2f} giây (~{duration_sec/3600:.2f} giờ)")

Khởi tạo B1 từ yolov8n-p2.yaml và nạp pretrained backbone từ yolov8n.pt...
Transferred 219/437 items from pretrained weights
=== BẮT ĐẦU HUẤN LUYỆN B1 FULL (20 EPOCHS, BATCH 16) ===
New https://pypi.org/project/ultralytics/8.4.160 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.153 🚀 Python-3.13.5 torch-2.14.0 CPU (Apple M1 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/Project/kltn-pcb/data/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_

### 8. Phương án dự phòng xử lý quá tải bộ nhớ VRAM
Nếu xảy ra lỗi tràn bộ nhớ (Out of Memory) do tầng $P_2$ ($160 \times 160$) tiêu tốn nhiều bộ nhớ gradient hơn:
- Hạ batch size xuống 8 hoặc 4 làm phương án xử lý cuối cùng.
- Ghi nhận rõ batch cũ, batch mới và limitation so với $B_0$ vào `docs/decision_log.md`.

In [13]:
if not training_success:
    FALLBACK_BATCH = 8
    print(f"KÍCH HOẠT FALLBACK BATCH: Giảm từ {BATCH_SIZE} xuống {FALLBACK_BATCH}...")
    
    # Ghi nhận vào decision_log.md
    vram_log = f"""\n## {datetime.now().strftime('%Y-%m-%d')}: Điều Chỉnh Batch Size B1 Do VRAM Constraint\n- Quyết định: Giảm batch size huấn luyện B1 từ {BATCH_SIZE} xuống {FALLBACK_BATCH}.\n- Lý do: Tầng P2 độ phân giải 160x160 làm tăng bộ nhớ gradient vượt quá dung lượng GPU/MPS hiện tại.\n- Limitation: Sự khác biệt về batch size (B0: {BATCH_SIZE} vs B1: {FALLBACK_BATCH}) có thể ảnh hưởng nhỏ tới variance của gradient, cần lưu ý khi thảo luận báo cáo.\n"""
    with open(ROOT / "docs" / "decision_log.md", "a", encoding="utf-8") as f:
        f.write(vram_log)
    
    start_fallback_time = time.time()
    model_b1 = YOLO(str(P2_CONFIG_PATH)).load(str(PRETRAINED_WEIGHTS))
    train_results = model_b1.train(
        data=str(DATA_YAML_PATH),
        epochs=EPOCHS,
        imgsz=int(base_cfg.get("imgsz", 640)),
        batch=FALLBACK_BATCH,
        seed=int(base_cfg.get("seed", 41)),
        optimizer=str(base_cfg.get("optimizer", "auto")),
        patience=int(base_cfg.get("patience", 30)),
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        degrees=0.0,
        shear=0.0,
        perspective=0.0,
        flipud=0.0,
        fliplr=0.5,
        mosaic=1.0,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        scale=0.5,
        translate=0.1,
        plots=True
    )
    BATCH_SIZE = FALLBACK_BATCH
    duration_sec = time.time() - start_fallback_time
    print(f"Hoàn thành huấn luyện fallback trong {duration_sec:.2f} giây!")
else:
    print("Huấn luyện thành công với cấu hình batch size chuẩn (batch = 16), không cần kích hoạt fallback.")

Huấn luyện thành công với cấu hình batch size chuẩn (batch = 16), không cần kích hoạt fallback.


### 9. Thẩm định mô hình sau huấn luyện (Validation Check)
Đánh giá checkpoint tốt nhất (`runs/B1/weights/best.pt`) trên tập dữ liệu kiểm định `val` (`data/splits/val.txt`).

In [14]:
best_weights_path = RUN_DIR / "weights" / "best.pt"
assert best_weights_path.exists(), f"LỖI: Không tìm thấy checkpoint: {best_weights_path}"

eval_model = YOLO(str(best_weights_path))
print(f"Đang thẩm định checkpoint {best_weights_path.name} trên validation split...")
val_metrics = eval_model.val(data=str(DATA_YAML_PATH), split="val", imgsz=640, batch=BATCH_SIZE)

print("=== KẾT QUẢ THẨM ĐỊNH SƠ BỘ B1 TRÊN VALIDATION SET ===")
print(f"  - mAP@0.5      : {val_metrics.box.map50:.5f}")
print(f"  - mAP@0.5:0.95 : {val_metrics.box.map:.5f}")
print(f"  - Precision    : {val_metrics.box.mp:.5f}")
print(f"  - Recall       : {val_metrics.box.mr:.5f}")

Đang thẩm định checkpoint best.pt trên validation split...
Ultralytics 8.4.153 🚀 Python-3.13.5 torch-2.14.0 CPU (Apple M1 Pro)
YOLOv8n-p2 summary (fused): 90 layers, 2,921,832 parameters, 0 gradients, 12.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1968.8±1765.3 MB/s, size: 968.1 KB)
val: Scanning data/raw/PKU-Market-PCB(Data enhanced version)/train/labels.cache... 368 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 368/368 308.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 4.3s/it 1:404.2sss
                   all        368       1860      0.712      0.671      0.681      0.273
          missing_hole         60        300       0.51      0.877      0.778      0.247
            mouse_bite         60        304      0.989      0.588      0.724      0.334
          open_circuit         64        324      0.589      0.571      0.481      0.203
                 short         60        304     

### 10. Trực quan hóa dự đoán mẫu (Sample Predictions)
Chạy dự đoán trên một số ảnh mẫu từ `data/splits/val.txt` và lưu ảnh trực quan hóa vào `results/B1/predictions/`.

In [15]:
pred_dir = RESULT_DIR / "predictions"
pred_dir.mkdir(parents=True, exist_ok=True)

val_lines = [l.strip() for l in VAL_TXT.read_text(encoding="utf-8").splitlines() if l.strip()]
sample_imgs = []
for l in val_lines[:5]:
    p = Path(l) if Path(l).is_absolute() else (ROOT / l)
    if p.exists():
        sample_imgs.append(str(p))

print(f"Chạy dự đoán mẫu trên {len(sample_imgs)} ảnh validation...")
preds = eval_model.predict(source=sample_imgs, imgsz=640, conf=0.25, save=True, project=str(RESULT_DIR), name="predictions", exist_ok=True)
print(f"Đã lưu ảnh trực quan hóa kết quả dự đoán vào: {pred_dir}")

Chạy dự đoán mẫu trên 5 ảnh validation...

0: 640x640 5 missing_holes, 1 open_circuit, 1 spur, 69.7ms
1: 640x640 5 missing_holes, 1 open_circuit, 2 spurs, 69.7ms
2: 640x640 5 missing_holes, 69.7ms
3: 640x640 5 missing_holes, 1 open_circuit, 1 spur, 69.7ms
4: 640x640 5 missing_holes, 1 open_circuit, 1 spur, 69.7ms
Speed: 1.6ms preprocess, 69.7ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /Users/Project/kltn-pcb/results/B1/predictions
Đã lưu ảnh trực quan hóa kết quả dự đoán vào: /Users/Project/kltn-pcb/results/B1/predictions


### 11. Thẩm định các file đầu ra bắt buộc của B1
Xác nhận sự tồn tại và tính hợp lệ của:
- `runs/B1/results.csv`
- `runs/B1/weights/best.pt`
- `runs/B1/weights/last.pt`
- `results/B1/predictions/`

In [16]:
results_csv_path = RUN_DIR / "results.csv"
best_pt_path = RUN_DIR / "weights" / "best.pt"
last_pt_path = RUN_DIR / "weights" / "last.pt"

assert results_csv_path.exists(), f"LỖI: Thiếu {results_csv_path}"
assert best_pt_path.exists(), f"LỖI: Thiếu {best_pt_path}"
assert last_pt_path.exists(), f"LỖI: Thiếu {last_pt_path}"

print("=== KIỂM TRA ĐẦU RA B1 ===")
print(f"  [PASS] results.csv   : {results_csv_path} ({results_csv_path.stat().st_size} bytes)")
print(f"  [PASS] best.pt       : {best_pt_path} ({best_pt_path.stat().st_size / (1024*1024):.2f} MB)")
print(f"  [PASS] last.pt       : {last_pt_path} ({last_pt_path.stat().st_size / (1024*1024):.2f} MB)")
print(f"  [PASS] predictions   : {pred_dir} (tồn tại và sẵn sàng)")

=== KIỂM TRA ĐẦU RA B1 ===
  [PASS] results.csv   : /Users/Project/kltn-pcb/runs/B1/results.csv (2557 bytes)
  [PASS] best.pt       : /Users/Project/kltn-pcb/runs/B1/weights/best.pt (5.93 MB)
  [PASS] last.pt       : /Users/Project/kltn-pcb/runs/B1/weights/last.pt (5.93 MB)
  [PASS] predictions   : /Users/Project/kltn-pcb/results/B1/predictions (tồn tại và sẵn sàng)


### 12. Cập nhật metadata hoàn chỉnh (`runs/B1/run_metadata.json`)

In [17]:
run_meta["end_time"] = datetime.now(timezone.utc).isoformat()
run_meta["duration_seconds"] = round(duration_sec, 2)
run_meta["duration_hours"] = round(duration_sec / 3600, 2)
run_meta["batch"] = BATCH_SIZE
run_meta["best_checkpoint"] = str(best_pt_path.relative_to(ROOT))
run_meta["last_checkpoint"] = str(last_pt_path.relative_to(ROOT))
run_meta["val_mAP50"] = float(round(val_metrics.box.map50, 5))
run_meta["val_mAP50_95"] = float(round(val_metrics.box.map, 5))

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(run_meta, f, indent=2)

print(f"Đã cập nhật run metadata hoàn chỉnh tại: {meta_path}")
print(json.dumps(run_meta, indent=2))

Đã cập nhật run metadata hoàn chỉnh tại: /Users/Project/kltn-pcb/runs/B1/run_metadata.json
{
  "experiment": "B1",
  "mode": "full_B1_P2",
  "start_time": "2026-09-23T04:52:53.796401+00:00",
  "end_time": "2026-09-23T19:42:08.163138+00:00",
  "duration_seconds": 53252.37,
  "git_commit": "31667dedef749a9acccca5832c47868cb39cd9e6",
  "base_model": "configs/yolov8n-p2.yaml",
  "seed": 41,
  "epochs": 20,
  "imgsz": 640,
  "batch": 16,
  "augmentation_recipe": {
    "degrees": 0.0,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "scale": 0.5,
    "translate": 0.1
  },
  "duration_hours": 14.79,
  "best_checkpoint": "runs/B1/weights/best.pt",
  "last_checkpoint": "runs/B1/weights/last.pt",
  "val_mAP50": 0.68123,
  "val_mAP50_95": 0.27283
}


### 13. Ghi nhận nhật ký quyết định kỹ thuật (`docs/decision_log.md`)
Lưu trữ bằng chứng hoàn tất huấn luyện $B_1$, ghi nhận các thông số kiểm chứng và tái khẳng định nguyên tắc không dùng tập test để tuning.

In [18]:
decision_log_path = ROOT / "docs" / "decision_log.md"
now_date = datetime.now().strftime("%Y-%m-%d")

with open(SPLIT_MANIFEST, "r", encoding="utf-8") as f:
    manifest_data = json.load(f)

train_md5 = manifest_data.get("splits", {}).get("train", {}).get("md5", "unknown")
val_md5 = manifest_data.get("splits", {}).get("val", {}).get("md5", "unknown")
test_md5 = manifest_data.get("splits", {}).get("test", {}).get("md5", "unknown")

log_entry = f"""
## {now_date}: Huấn Luyện Thành Công Mô Hình B1 (YOLOv8n + P2 Full Train)

- Quyết định: Hoàn tất huấn luyện mô hình B1 (YOLOv8n-P2) phục vụ đánh giá năng lực phát hiện khuyết tật vi mô so với Baseline B0.
- Cấu hình huấn luyện: model=configs/yolov8n-p2.yaml, pretrained=yolov8n.pt, imgsz=640, epochs={EPOCHS}, batch={BATCH_SIZE}, optimizer=auto, patience=30.
- Seed đã khóa: {base_cfg.get("seed", 41)}.
- Split Checksum (MD5): train={train_md5}, val={val_md5}, test={test_md5}.
- Augmentation Recipe: degrees=0.0, shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.5, mosaic=1.0, scale=0.5, translate=0.1, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4.
- Thời lượng huấn luyện: {duration_sec:.2f} s (~{duration_sec/3600:.2f} giờ).
- Checkpoint tốt nhất: `runs/B1/weights/best.pt` (val mAP@0.5: {val_metrics.box.map50:.4f}, mAP@0.5:0.95: {val_metrics.box.map:.4f}).
- Ràng buộc kiểm chứng: Mô hình B1 **chỉ thay đổi duy nhất biến kiến trúc P2**; tuyệt đối không sử dụng tập test (`data/splits/test.txt`) để tuning bất kỳ siêu tham số nào.
"""

with open(decision_log_path, "a", encoding="utf-8") as f:
    f.write(log_entry)

print(f"Đã append thành công ghi nhận huấn luyện B1 vào {decision_log_path}")

Đã append thành công ghi nhận huấn luyện B1 vào /Users/Project/kltn-pcb/docs/decision_log.md
